# AML Regulatory Document — Parsing & Chunking Pipeline

Production-grade hierarchical chunker for dense regulatory/legal PDFs (e.g., FATF guidance, AML policy documents).

**Design goals:**
1. Preserve document hierarchy (Section → Sub-section → Clause)
2. Keep tables, boxed case studies, and lists atomic
3. Produce parent/child chunks for parent-document retrieval
4. Resolve cross-references (R.1, INR.1, Section 3, Box 7, Table 2, etc.)
5. Emit rich metadata for filtering and audit
6. **Parallel processing** at page-parse and chunk-build stages

**Parser strategy:**
- Primary: **PyMuPDF** for fast layout-aware text extraction + **pdfplumber** for tables (run in parallel)
- In production, swap for **Docling** (IBM) — best hierarchy preservation
- Tokenizer: `tiktoken cl100k_base` (matches OpenAI embeddings)

**Parallelism strategy:**
- `ProcessPoolExecutor` for page parsing (CPU-bound, GIL-released by C extensions)
- `ThreadPoolExecutor` for chunk assembly (I/O-light, thread-safe)
- `imap_unordered` for streaming progress with `tqdm`


In [1]:
# Run once in a fresh environment:
# !pip install pymupdf pdfplumber tiktoken tqdm pandas --quiet

# Optional (production): swap PyMuPDF for Docling
# !pip install docling --quiet


In [2]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import uuid
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from enum import Enum
from pathlib import Path
from typing import Iterator

import fitz  # PyMuPDF
import pdfplumber
import tiktoken
from tqdm.auto import tqdm

# ---- Chunking parameters (tuned for dense AML/regulatory text) ----
CHILD_CHUNK_TOKENS      = 400    # target size for embedded chunks
CHILD_CHUNK_OVERLAP_PCT = 0.15   # 60 tokens at 400 target
PARENT_CHUNK_TOKENS     = 1500   # context returned to LLM at generation time
PARENT_CHUNK_OVERLAP_PCT = 0.10
MIN_CHUNK_TOKENS        = 50     # below this → flagged as suspicious
MAX_CHUNK_TOKENS        = 800    # hard ceiling

# ---- Parallelism ----
# Use physical core count; override via env for tuning
N_WORKERS = int(os.getenv("CHUNKER_WORKERS", max(1, (os.cpu_count() or 4) - 1)))

# ---- Tokenizer (OpenAI-compatible, with offline fallback) ----
# In production use tiktoken's cl100k_base (matches OpenAI embeddings exactly).
# In air-gapped environments (common in compliance), fall back gracefully.
class _FallbackTokenizer:
    """Approximate tokenizer for environments without tiktoken network access."""
    def encode(self, text, **kwargs):
        # Approx 1 token per 4 characters (±8% vs cl100k_base on English prose)
        return list(range(max(1, len(text) // 4)))
    def decode(self, ids, **kwargs):
        return ""

try:
    TOKENIZER = tiktoken.get_encoding("cl100k_base")
    _USING_TIKTOKEN = True
except Exception:
    TOKENIZER = _FallbackTokenizer()
    _USING_TIKTOKEN = False

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("chunker")

print(f"Workers: {N_WORKERS}")
print(f"Tokenizer: {'tiktoken cl100k_base' if _USING_TIKTOKEN else 'char-based fallback (approximate)'}")
print(f"Child chunk target: {CHILD_CHUNK_TOKENS} tokens ({int(CHILD_CHUNK_TOKENS*CHILD_CHUNK_OVERLAP_PCT)} overlap)")
print(f"Parent chunk target: {PARENT_CHUNK_TOKENS} tokens ({int(PARENT_CHUNK_TOKENS*PARENT_CHUNK_OVERLAP_PCT)} overlap)")


Workers: 1
Tokenizer: char-based fallback (approximate)
Child chunk target: 400 tokens (60 overlap)
Parent chunk target: 1500 tokens (150 overlap)


In [3]:
class ChunkType(str, Enum):
    PROSE      = "prose"
    HEADING    = "heading"
    TABLE      = "table"
    LIST       = "list"
    BOX        = "box"          # boxed case studies / examples
    DEFINITION = "definition"
    FOOTNOTE   = "footnote"


@dataclass
class Block:
    """Raw structural block extracted from a page, pre-chunking."""
    block_type: ChunkType
    text: str
    page_start: int
    page_end: int
    # Optional structural hints
    heading_level: int | None = None          # 1..6 for headings
    section_path: list[str] = field(default_factory=list)
    table_html: str | None = None             # for table blocks
    raw_rows: list[list[str]] | None = None   # for table blocks


@dataclass
class Chunk:
    """Final chunk ready for embedding and indexing."""
    chunk_id: str
    doc_id: str
    text: str
    # Structural
    chunk_type: ChunkType
    section_path: list[str]
    hierarchy_level: int
    page_start: int
    page_end: int
    # Parent/child linkage
    parent_chunk_id: str | None = None
    child_chunk_ids: list[str] = field(default_factory=list)
    # Token accounting
    token_count: int = 0
    # Semantic / extracted (populated later by LLM stage)
    regulatory_references: list[str] = field(default_factory=list)
    topic_tags: list[str] = field(default_factory=list)
    # Operational
    doc_title: str = ""
    doc_version: str = ""
    doc_authority: str = ""
    ingested_at: str = ""
    chunk_hash: str = ""
    parser_version: str = "pymupdf-1.x"

    def to_dict(self) -> dict:
        d = asdict(self)
        d["chunk_type"] = self.chunk_type.value
        return d


In [4]:
def count_tokens(text: str) -> int:
    """Cheap token count via tiktoken."""
    return len(TOKENIZER.encode(text, disallowed_special=()))


def clean_text(text: str) -> str:
    """Normalize whitespace without destroying intentional line breaks."""
    # Collapse >2 consecutive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Collapse horizontal whitespace runs
    text = re.sub(r"[ \t]+", " ", text)
    # Strip trailing spaces on each line
    text = re.sub(r" +\n", "\n", text)
    # Remove common PDF artifacts
    text = text.replace("\x0c", "\n")  # form feed
    return text.strip()


def sha256_short(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


In [5]:
# Heading patterns tailored to FATF / AML policy documents.
# Tune per document family after reviewing 5-10 sample pages.

HEADING_PATTERNS = [
    # Level 1: "Section 1:", "Section 2:", "Annex A.", "Annex B."
    (re.compile(r"^(Section\s+\d+|Annex\s+[A-Z])[\.\:]?\s+(.+)$"), 1),
    # Level 2: Major topic inside a section (title-case, short, no sentence punctuation)
    (re.compile(r"^([A-Z][A-Za-z][A-Za-z\s\-/&,]{4,80})$"), 2),
    # Level 3: Subsection title (shorter)
    (re.compile(r"^([A-Z][a-z]+(?:\s+[A-Za-z][a-z]+){1,6})$"), 3),
]

# Boxes and tables have distinctive prefixes in FATF docs
BOX_PATTERN   = re.compile(r"^Box\s+\d+[\.\:]", re.IGNORECASE)
TABLE_PATTERN = re.compile(r"^Table\s+\d+[\.\:]", re.IGNORECASE)


def classify_line(line: str) -> tuple[ChunkType | None, int | None]:
    """Return (chunk_type, heading_level) for a line if it looks structural."""
    s = line.strip()
    if not s:
        return None, None

    if BOX_PATTERN.match(s):
        return ChunkType.BOX, 2
    if TABLE_PATTERN.match(s):
        return ChunkType.TABLE, 2

    for pattern, level in HEADING_PATTERNS:
        m = pattern.match(s)
        if not m:
            continue
        # Anti-false-positive filters
        if len(s) > 120:
            continue
        if s.endswith((".", "?", "!")) and level > 1:
            continue
        if len(s.split()) < 2 and level > 1:
            continue
        return ChunkType.HEADING, level

    return None, None


In [6]:
# IMPORTANT: functions passed to ProcessPoolExecutor must be picklable
# (defined at module level, not nested). In a notebook, this means the cell
# must be executed before the pool is created.

def parse_page_pymupdf(args: tuple[str, int]) -> dict:
    """
    Extract text blocks from a single page.
    Returns a serializable dict (workers can't share live objects).
    """
    pdf_path, page_idx = args
    doc = fitz.open(pdf_path)
    try:
        page = doc[page_idx]
        # "blocks" gives us layout-aware groupings; sorted top-to-bottom, left-to-right
        raw_blocks = page.get_text("blocks", sort=True)
        out = []
        for b in raw_blocks:
            # b = (x0, y0, x1, y1, text, block_no, block_type)
            text = (b[4] or "").strip()
            if not text:
                continue
            out.append({
                "text": text,
                "bbox": [b[0], b[1], b[2], b[3]],
                "page": page_idx + 1,  # 1-indexed for humans
            })
        return {"page": page_idx + 1, "blocks": out}
    finally:
        doc.close()


def parse_tables_pdfplumber(args: tuple[str, int]) -> dict:
    """
    Extract tables from a single page using pdfplumber.
    Separated from text parsing so both run in parallel.
    """
    pdf_path, page_idx = args
    tables_out = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            page = pdf.pages[page_idx]
            raw_tables = page.extract_tables() or []
            for t_idx, table in enumerate(raw_tables):
                if not table or len(table) < 2:
                    continue
                # Serialize to markdown for LLM consumption
                rows = [[(c or "").strip() for c in row] for row in table]
                md = _rows_to_markdown(rows)
                tables_out.append({
                    "page": page_idx + 1,
                    "table_index": t_idx,
                    "rows": rows,
                    "markdown": md,
                })
    except Exception as e:
        # pdfplumber occasionally fails on complex tables — log, don't crash
        return {"page": page_idx + 1, "tables": [], "error": str(e)}
    return {"page": page_idx + 1, "tables": tables_out}


def _rows_to_markdown(rows: list[list[str]]) -> str:
    if not rows:
        return ""
    header = rows[0]
    body   = rows[1:] if len(rows) > 1 else []
    md = "| " + " | ".join(header) + " |\n"
    md += "| " + " | ".join(["---"] * len(header)) + " |\n"
    for r in body:
        # Pad row if short
        r = r + [""] * (len(header) - len(r))
        md += "| " + " | ".join(r[:len(header)]) + " |\n"
    return md.strip()


In [7]:
def parse_pdf_parallel(pdf_path: str | Path, n_workers: int = N_WORKERS) -> dict:
    """
    Parse all pages of a PDF in parallel.
    Returns dict with 'pages' (text blocks) and 'tables' (extracted tables).
    """
    pdf_path = str(pdf_path)
    with fitz.open(pdf_path) as doc:
        n_pages = doc.page_count

    log.info(f"Parsing {n_pages} pages with {n_workers} workers")

    # Build task lists
    text_tasks  = [(pdf_path, i) for i in range(n_pages)]
    table_tasks = [(pdf_path, i) for i in range(n_pages)]

    pages_result  = [None] * n_pages
    tables_result = [None] * n_pages

    # Run text extraction and table extraction in parallel.
    # Two separate pools isolate failures and let both stages stream progress.
    with ProcessPoolExecutor(max_workers=n_workers) as exe:
        # Text extraction
        futures = {exe.submit(parse_page_pymupdf, t): t[1] for t in text_tasks}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Text"):
            page_idx = futures[fut]
            try:
                pages_result[page_idx] = fut.result()
            except Exception as e:
                log.warning(f"Page {page_idx+1} text extraction failed: {e}")
                pages_result[page_idx] = {"page": page_idx+1, "blocks": [], "error": str(e)}

        # Table extraction
        futures = {exe.submit(parse_tables_pdfplumber, t): t[1] for t in table_tasks}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Tables"):
            page_idx = futures[fut]
            try:
                tables_result[page_idx] = fut.result()
            except Exception as e:
                log.warning(f"Page {page_idx+1} table extraction failed: {e}")
                tables_result[page_idx] = {"page": page_idx+1, "tables": [], "error": str(e)}

    return {
        "n_pages": n_pages,
        "pages":  pages_result,
        "tables": tables_result,
    }


In [8]:
def _block_intersects_any(bbox: list[float], tables_on_page: list[dict]) -> bool:
    """
    Tables extracted by pdfplumber overlap with PyMuPDF text blocks.
    Rudimentary check — in production use real bbox intersection.
    """
    # pdfplumber doesn't surface bbox in the format above; here we just dedupe
    # by checking if the block text appears to be table-like (pipe-heavy).
    return False


def assemble_blocks(parsed: dict) -> list[Block]:
    """
    Flatten parsed output into a linear sequence of Block objects,
    maintaining reading order and tagging structural role.
    """
    blocks: list[Block] = []
    current_section_path: list[str] = []

    # First: absorb tables as atomic blocks, keyed by page
    tables_by_page: dict[int, list[dict]] = {}
    for t_page in parsed["tables"]:
        if t_page and t_page.get("tables"):
            tables_by_page[t_page["page"]] = t_page["tables"]

    # Walk pages in order
    for page_data in parsed["pages"]:
        if not page_data or not page_data.get("blocks"):
            continue

        page_num = page_data["page"]

        # Emit table blocks first for this page (with surrounding context preserved)
        for table in tables_by_page.get(page_num, []):
            md = table["markdown"]
            if not md:
                continue
            blocks.append(Block(
                block_type=ChunkType.TABLE,
                text=md,
                page_start=page_num,
                page_end=page_num,
                section_path=list(current_section_path),
                table_html=md,  # simple markdown rendering
                raw_rows=table["rows"],
            ))

        # Now text blocks
        for b in page_data["blocks"]:
            text = clean_text(b["text"])
            if not text:
                continue

            # Heuristic: skip single-line page numbers / repeating headers/footers
            if re.fullmatch(r"\d{1,4}", text):
                continue
            if re.fullmatch(r"(Money Laundering National Risk Assessment Guidance.*)", text):
                continue
            # Skip short all-caps fragments (acronyms in glossary columns,
            # headers that leaked into body). These pollute chunks when a
            # multi-column layout causes acronym lists to fragment.
            if len(text) < 30 and re.fullmatch(r"[A-Z][A-Z0-9/\-]{1,8}", text.strip()):
                continue
            # Skip URLs that leak in from footnotes
            if re.fullmatch(r"https?://\S+", text.strip()):
                continue

            # Walk lines to detect headings and boxes
            lines = text.split("\n")
            buffer_lines: list[str] = []
            for line in lines:
                ctype, level = classify_line(line)

                if ctype == ChunkType.HEADING:
                    # Flush buffered prose
                    if buffer_lines:
                        prose = "\n".join(buffer_lines).strip()
                        if prose:
                            blocks.append(Block(
                                block_type=ChunkType.PROSE,
                                text=prose,
                                page_start=page_num,
                                page_end=page_num,
                                section_path=list(current_section_path),
                            ))
                        buffer_lines = []
                    # Update section path based on level
                    current_section_path = current_section_path[: max(0, level - 1)]
                    current_section_path.append(line.strip())
                    blocks.append(Block(
                        block_type=ChunkType.HEADING,
                        text=line.strip(),
                        page_start=page_num,
                        page_end=page_num,
                        heading_level=level,
                        section_path=list(current_section_path),
                    ))
                elif ctype == ChunkType.BOX:
                    # Flush buffered prose
                    if buffer_lines:
                        prose = "\n".join(buffer_lines).strip()
                        if prose:
                            blocks.append(Block(
                                block_type=ChunkType.PROSE,
                                text=prose,
                                page_start=page_num,
                                page_end=page_num,
                                section_path=list(current_section_path),
                            ))
                        buffer_lines = []
                    # Start a box — consume subsequent lines until next heading/box
                    buffer_lines.append(line.strip())
                else:
                    buffer_lines.append(line)

            # Flush remaining
            if buffer_lines:
                prose = "\n".join(buffer_lines).strip()
                if prose:
                    blocks.append(Block(
                        block_type=ChunkType.PROSE,
                        text=prose,
                        page_start=page_num,
                        page_end=page_num,
                        section_path=list(current_section_path),
                    ))

    log.info(f"Assembled {len(blocks)} raw blocks "
             f"({sum(1 for b in blocks if b.block_type==ChunkType.HEADING)} headings, "
             f"{sum(1 for b in blocks if b.block_type==ChunkType.TABLE)} tables, "
             f"{sum(1 for b in blocks if b.block_type==ChunkType.PROSE)} prose, "
             f"{sum(1 for b in blocks if b.block_type==ChunkType.BOX)} boxes)")
    return blocks


In [9]:
# Cross-reference extraction
REF_PATTERNS = [
    re.compile(r"\bR\.\s*\d+(?:[./]\d+)?\b"),
    re.compile(r"\bINR\.\s*\d+(?:[./]\d+)?\b"),
    re.compile(r"\bIO\.\s*\d+\b"),
    re.compile(r"\bSection\s+\d+\b"),
    re.compile(r"\bBox\s+\d+\b"),
    re.compile(r"\bTable\s+\d+\b"),
    re.compile(r"\bAnnex\s+[A-Z]\b"),
    re.compile(r"\bFATF\b"),
    re.compile(r"\bFIU\b"),
]


def extract_references(text: str) -> list[str]:
    found: set[str] = set()
    for p in REF_PATTERNS:
        for m in p.findall(text):
            found.add(m.strip())
    return sorted(found)


def split_by_tokens(text: str, target: int, overlap: int) -> list[str]:
    """
    Token-aware splitter that respects sentence boundaries where possible.
    Overlap is applied as a sentence-aligned tail.
    """
    if count_tokens(text) <= target:
        return [text]

    # Split to sentences (simple heuristic — OK for legal/regulatory prose)
    sentences = re.split(r"(?<=[\.\?\!])\s+(?=[A-Z])", text)

    chunks: list[str] = []
    current: list[str] = []
    current_tokens = 0

    for sent in sentences:
        sent_tokens = count_tokens(sent)
        # Oversized single sentence — fall back to hard split
        if sent_tokens > target:
            if current:
                chunks.append(" ".join(current).strip())
                current, current_tokens = [], 0
            # Hard split: if tiktoken, split on tokens; else, split on characters
            if _USING_TIKTOKEN:
                ids = TOKENIZER.encode(sent, disallowed_special=())
                for i in range(0, len(ids), target):
                    chunks.append(TOKENIZER.decode(ids[i:i+target]))
            else:
                # char-based fallback: target tokens ≈ target * 4 chars
                char_window = target * 4
                for i in range(0, len(sent), char_window):
                    chunks.append(sent[i:i+char_window])
            continue

        if current_tokens + sent_tokens <= target:
            current.append(sent)
            current_tokens += sent_tokens
        else:
            chunks.append(" ".join(current).strip())
            # Build overlap: keep trailing sentences up to `overlap` tokens
            tail, tail_tokens = [], 0
            for s in reversed(current):
                t = count_tokens(s)
                if tail_tokens + t > overlap:
                    break
                tail.insert(0, s)
                tail_tokens += t
            current = tail + [sent]
            current_tokens = tail_tokens + sent_tokens

    if current:
        chunks.append(" ".join(current).strip())
    return chunks


def build_chunks_from_group(
    group: list[Block],
    doc_id: str,
    doc_meta: dict,
) -> list[Chunk]:
    """
    Convert a contiguous group of blocks under one section into parent+child chunks.
    Atomic blocks (tables, boxes) stay as single chunks.
    """
    if not group:
        return []

    results: list[Chunk] = []
    section_path = group[0].section_path
    hierarchy_level = len(section_path)
    page_start = group[0].page_start
    page_end = group[-1].page_end
    now = datetime.now(timezone.utc).isoformat()

    # 1. Emit atomic (table/box) blocks as single chunks — DO NOT split these
    atomic_blocks = [b for b in group if b.block_type in (ChunkType.TABLE, ChunkType.BOX)]
    prose_blocks  = [b for b in group if b.block_type not in (ChunkType.TABLE, ChunkType.BOX, ChunkType.HEADING)]

    for ab in atomic_blocks:
        ctx_header = " > ".join(ab.section_path) if ab.section_path else ""
        text = (f"[{ab.block_type.value.upper()} in: {ctx_header}]\n\n" + ab.text) if ctx_header else ab.text
        chunk_id = str(uuid.uuid4())
        results.append(Chunk(
            chunk_id=chunk_id,
            doc_id=doc_id,
            text=text,
            chunk_type=ab.block_type,
            section_path=ab.section_path,
            hierarchy_level=len(ab.section_path),
            page_start=ab.page_start,
            page_end=ab.page_end,
            token_count=count_tokens(text),
            regulatory_references=extract_references(text),
            doc_title=doc_meta.get("title", ""),
            doc_version=doc_meta.get("version", ""),
            doc_authority=doc_meta.get("authority", ""),
            ingested_at=now,
            chunk_hash=sha256_short(text),
        ))

    if not prose_blocks:
        return results

    # 2. Concatenate prose in order, split into parent chunks
    concatenated = "\n\n".join(b.text for b in prose_blocks).strip()
    if not concatenated:
        return results

    parent_overlap = int(PARENT_CHUNK_TOKENS * PARENT_CHUNK_OVERLAP_PCT)
    parent_texts = split_by_tokens(concatenated, PARENT_CHUNK_TOKENS, parent_overlap)

    child_overlap = int(CHILD_CHUNK_TOKENS * CHILD_CHUNK_OVERLAP_PCT)

    for parent_text in parent_texts:
        parent_id = str(uuid.uuid4())
        parent_chunk = Chunk(
            chunk_id=parent_id,
            doc_id=doc_id,
            text=parent_text,
            chunk_type=ChunkType.PROSE,
            section_path=section_path,
            hierarchy_level=hierarchy_level,
            page_start=page_start,
            page_end=page_end,
            token_count=count_tokens(parent_text),
            regulatory_references=extract_references(parent_text),
            doc_title=doc_meta.get("title", ""),
            doc_version=doc_meta.get("version", ""),
            doc_authority=doc_meta.get("authority", ""),
            ingested_at=now,
            chunk_hash=sha256_short(parent_text),
        )

        # Children
        child_texts = split_by_tokens(parent_text, CHILD_CHUNK_TOKENS, child_overlap)
        child_ids = []
        child_chunks = []
        for ct in child_texts:
            cid = str(uuid.uuid4())
            child_ids.append(cid)
            # Prepend section breadcrumb to improve embedding context
            breadcrumb = " > ".join(section_path) if section_path else ""
            enriched = f"[Context: {breadcrumb}]\n\n{ct}" if breadcrumb else ct
            child_chunks.append(Chunk(
                chunk_id=cid,
                doc_id=doc_id,
                text=enriched,
                chunk_type=ChunkType.PROSE,
                section_path=section_path,
                hierarchy_level=hierarchy_level,
                page_start=page_start,
                page_end=page_end,
                parent_chunk_id=parent_id,
                token_count=count_tokens(enriched),
                regulatory_references=extract_references(enriched),
                doc_title=doc_meta.get("title", ""),
                doc_version=doc_meta.get("version", ""),
                doc_authority=doc_meta.get("authority", ""),
                ingested_at=now,
                chunk_hash=sha256_short(enriched),
            ))
        parent_chunk.child_chunk_ids = child_ids
        results.append(parent_chunk)
        results.extend(child_chunks)

    return results


def group_blocks_by_section(blocks: list[Block]) -> list[list[Block]]:
    """Group contiguous blocks that share the same section_path."""
    groups: list[list[Block]] = []
    current: list[Block] = []
    current_path: list[str] | None = None

    for b in blocks:
        if b.block_type == ChunkType.HEADING:
            # Heading marks a boundary AND contributes its own line to the NEXT group
            if current:
                groups.append(current)
            current = []
            current_path = b.section_path
            continue
        if current_path is None:
            current_path = b.section_path
        if b.section_path != current_path:
            if current:
                groups.append(current)
            current = [b]
            current_path = b.section_path
        else:
            current.append(b)

    if current:
        groups.append(current)
    return groups


In [10]:
def build_all_chunks_parallel(
    blocks: list[Block],
    doc_id: str,
    doc_meta: dict,
    n_workers: int = N_WORKERS,
) -> list[Chunk]:
    """
    Group blocks by section, build chunks for each group in parallel.

    ThreadPoolExecutor is used here rather than ProcessPoolExecutor because:
      1. The work is light (tokenization + string ops)
      2. Avoids pickling overhead of Block objects
      3. tiktoken releases GIL for the hot path
    """
    groups = group_blocks_by_section(blocks)
    log.info(f"Built {len(groups)} section groups; fanning out to {n_workers} threads")

    all_chunks: list[Chunk] = []
    with ThreadPoolExecutor(max_workers=n_workers) as exe:
        futures = [exe.submit(build_chunks_from_group, g, doc_id, doc_meta) for g in groups]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Chunking"):
            try:
                all_chunks.extend(fut.result())
            except Exception as e:
                log.error(f"Chunk building failed for a group: {e}")

    # Stable ordering by (page_start, then parent-before-children)
    all_chunks.sort(key=lambda c: (c.page_start, c.parent_chunk_id is not None))

    # Post-filter: drop chunks below the minimum token threshold that aren't
    # tables/boxes (which are atomic and intentionally preserved even if short).
    pre_filter = len(all_chunks)
    preserved_types = {ChunkType.TABLE, ChunkType.BOX, ChunkType.DEFINITION}
    kept = [c for c in all_chunks
            if c.token_count >= MIN_CHUNK_TOKENS or c.chunk_type in preserved_types]
    dropped = pre_filter - len(kept)
    if dropped:
        log.info(f"Filtered out {dropped} chunks below {MIN_CHUNK_TOKENS}-token threshold")

    # Repair dangling parent references after filtering
    kept_ids = {c.chunk_id for c in kept}
    for c in kept:
        if c.parent_chunk_id and c.parent_chunk_id not in kept_ids:
            c.parent_chunk_id = None  # orphan child becomes standalone
        c.child_chunk_ids = [cid for cid in c.child_chunk_ids if cid in kept_ids]

    log.info(f"Produced {len(kept)} total chunks "
             f"({sum(1 for c in kept if c.parent_chunk_id is None)} parents/atomic, "
             f"{sum(1 for c in kept if c.parent_chunk_id is not None)} children)")
    return kept


In [11]:
def process_document(
    pdf_path: str | Path,
    doc_meta: dict,
    output_jsonl: str | Path,
    n_workers: int = N_WORKERS,
) -> list[Chunk]:
    pdf_path = Path(pdf_path)
    output_jsonl = Path(output_jsonl)
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)

    doc_id = doc_meta.get("doc_id") or sha256_short(pdf_path.name)

    log.info(f"=== Processing {pdf_path.name} (doc_id={doc_id}) ===")

    # Stage 1: Parallel PDF parsing
    parsed = parse_pdf_parallel(pdf_path, n_workers=n_workers)

    # Stage 2: Block assembly (sequential — preserves reading order)
    blocks = assemble_blocks(parsed)

    # Stage 3: Parallel chunk building
    chunks = build_all_chunks_parallel(blocks, doc_id, doc_meta, n_workers=n_workers)

    # Stage 4: Persist JSONL
    with output_jsonl.open("w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c.to_dict(), ensure_ascii=False) + "\n")
    log.info(f"Wrote {len(chunks)} chunks → {output_jsonl}")

    return chunks


In [12]:
# Point to the FATF AML NRA guidance you uploaded
# Put the source PDF in this folder or set an absolute path (see README / chunking.md).
PDF_PATH = "Money-Laundering-National-Risk-Assessment-Guidance-2024.pdf"
OUTPUT_JSONL = "fatf_aml_nra_chunks.jsonl"

DOC_META = {
    "doc_id":    "fatf_ml_nra_2024",
    "title":     "Money Laundering National Risk Assessment Guidance",
    "version":   "November 2024",
    "authority": "FATF",
}

chunks = process_document(PDF_PATH, DOC_META, OUTPUT_JSONL)
print(f"\n✅ Done — {len(chunks)} chunks written to {OUTPUT_JSONL}")


2026-04-21 01:21:32,887 | INFO | === Processing Money-Laundering-National-Risk-Assessment-Guidance-2024_pdf_coredownload_inline_pdf.pdf (doc_id=fatf_ml_nra_2024) ===


2026-04-21 01:21:33,087 | INFO | Parsing 74 pages with 1 workers


Text:   0%|          | 0/74 [00:00<?, ?it/s]

Tables:   0%|          | 0/74 [00:00<?, ?it/s]

2026-04-21 01:22:02,055 | INFO | Assembled 963 raw blocks (196 headings, 31 tables, 736 prose, 0 boxes)


...

Chunking:   0%|          | 0/181 [00:00<?, ?it/s]

2026-04-21 01:22:02,375 | INFO | Filtered out 155 chunks below 50-token threshold


2026-04-21 01:22:02,377 | INFO | Produced 323 total chunks (141 parents/atomic, 182 children)


2026-04-21 01:22:02,423 | INFO | Wrote 323 chunks → /mnt/user-data/outputs/fatf_aml_nra_chunks.jsonl



✅ Done — 323 chunks written to /mnt/user-data/outputs/fatf_aml_nra_chunks.jsonl


## Sanity Checks & Visualization

The cells below let you visually and programmatically verify chunk quality. Run these after every ingestion.

**What to look for:**
- No orphaned headings (heading alone at the end of a chunk with < 50 tokens following)
- No fractured tables (incomplete markdown tables)
- Chunks fall within size bounds
- Section paths are plausible
- Cross-references are being extracted


In [13]:
import pandas as pd

def load_chunks_to_df(jsonl_path: str | Path) -> pd.DataFrame:
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    df["section_path_str"] = df["section_path"].apply(lambda p: " > ".join(p) if p else "(root)")
    df["n_references"] = df["regulatory_references"].apply(len)
    df["is_parent"] = df["parent_chunk_id"].isna()
    return df

df = load_chunks_to_df(OUTPUT_JSONL)
print(f"Loaded {len(df)} chunks")
df.head(3)


Loaded 323 chunks


,chunk_id,doc_id,text,chunk_type,section_path,hierarchy_level,page_start,page_end,parent_chunk_id,child_chunk_ids,...,topic_tags,doc_title,doc_version,doc_authority,ingested_at,chunk_hash,parser_version,section_path_str,n_references,is_parent
0,e3977b73-c47d-4663-9f47-a0f4839c3286,fatf_ml_nra_2024,Guidance\n\nNovember 2024\n\nThe Financial Act...,prose,"[FATF GUIDANCE, National Risk Assessment]",2,1,3,NaN,[d1dc3d52-0ccb-434e-b55b-086083765234],...,[],Money Laundering National Risk Assessment Guid...,November 2024,FATF,2026-04-21T01:22:02.058757+00:00,6b1e067d41852a67,pymupdf-1.x,FATF GUIDANCE > National Risk Assessment,1,True
1,d1dc3d52-0ccb-434e-b55b-086083765234,fatf_ml_nra_2024,[Context: FATF GUIDANCE > National Risk Assess...,prose,"[FATF GUIDANCE, National Risk Assessment]",2,1,3,e3977b73-c47d-4663-9f47-a0f4839c3286,[],...,[],Money Laundering National Risk Assessment Guid...,November 2024,FATF,2026-04-21T01:22:02.058757+00:00,4e5c8a96c5b91a8c,pymupdf-1.x,FATF GUIDANCE > National Risk Assessment,1,False
2,bf713eec-d0ee-4ddd-a65d-be7f1349759b,fatf_ml_nra_2024,Acronyms ........................................,prose,"[FATF GUIDANCE, Table of Contents]",2,3,4,NaN,"[932f6b46-4e7b-46d4-92f8-fea5eec28e24, da08c61...",...,[],Money Laundering National Risk Assessment Guid...,November 2024,FATF,2026-04-21T01:22:02.062049+00:00,0f4cf7c825b8dde4,pymupdf-1.x,FATF GUIDANCE > Table of Contents,6,True


In [14]:
print("=" * 70)
print("CHUNK SUMMARY")
print("=" * 70)
print(f"Total chunks:          {len(df)}")
print(f"  Parents/atomic:      {df['is_parent'].sum()}")
print(f"  Children:            {(~df['is_parent']).sum()}")
print()
print("By chunk type:")
print(df["chunk_type"].value_counts().to_string())
print()
print("Token count stats:")
print(df["token_count"].describe().round(1).to_string())
print()
print("Pages covered:")
print(f"  Min page: {df['page_start'].min()}")
print(f"  Max page: {df['page_end'].max()}")
print()
print("Cross-references found (top 15):")
all_refs = [r for refs in df["regulatory_references"] for r in refs]
print(pd.Series(all_refs).value_counts().head(15).to_string())


CHUNK SUMMARY
Total chunks:          323
  Parents/atomic:      141
  Children:            182

By chunk type:
chunk_type
prose    292
table     31

Token count stats:
count     323.0
mean      372.0
std       251.5
min        50.0
25%       223.0
50%       348.0
75%       412.5
max      1494.0

Pages covered:
  Min page: 1
  Max page: 74

Cross-references found (top 15):
Section 3    84
Section 1    70
FATF         65
Section 2    53
FIU          30
Annex A      23
INR.1         8
Annex B       6
R.1           6
R.2           6
Box 5         4
Box 8         4
Box 12        4
Box 18        4
Box 21        4


In [15]:
def run_sanity_checks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame of (check_name, passed, count, details).
    Designed to be run in CI as quality gates.
    """
    results = []

    # 1. Undersized chunks (< MIN_CHUNK_TOKENS) — often indicate parser errors
    under = df[df["token_count"] < MIN_CHUNK_TOKENS]
    results.append({
        "check": f"No chunks under {MIN_CHUNK_TOKENS} tokens",
        "passed": len(under) / max(1, len(df)) < 0.02,
        "count": len(under),
        "pct": round(100 * len(under) / max(1, len(df)), 2),
        "severity": "warn" if len(under) / max(1, len(df)) < 0.05 else "fail",
    })

    # 2. Oversized chunks (> MAX_CHUNK_TOKENS) — embedder truncation risk
    over = df[df["token_count"] > MAX_CHUNK_TOKENS]
    results.append({
        "check": f"No chunks over {MAX_CHUNK_TOKENS} tokens",
        "passed": len(over) == 0,
        "count": len(over),
        "pct": round(100 * len(over) / max(1, len(df)), 2),
        "severity": "fail" if len(over) > 0 else "pass",
    })

    # 3. Orphaned headings: any child chunk that is just a heading line
    orphans = df[(df["chunk_type"] == "heading") & (df["token_count"] < 20)]
    results.append({
        "check": "No orphaned headings",
        "passed": len(orphans) / max(1, len(df)) < 0.02,
        "count": len(orphans),
        "pct": round(100 * len(orphans) / max(1, len(df)), 2),
        "severity": "warn",
    })

    # 4. Fractured tables: table chunks without a valid markdown header separator
    tables = df[df["chunk_type"] == "table"]
    fractured = tables[~tables["text"].str.contains(r"\|\s*---", regex=True, na=False)]
    results.append({
        "check": "No fractured tables",
        "passed": len(fractured) == 0,
        "count": len(fractured),
        "pct": round(100 * len(fractured) / max(1, len(tables)), 2) if len(tables) else 0.0,
        "severity": "fail" if len(fractured) > 0 else "pass",
    })

    # 5. Parent/child integrity: every child must reference an existing parent
    parent_ids = set(df[df["is_parent"]]["chunk_id"])
    child_df   = df[~df["is_parent"]]
    dangling   = child_df[~child_df["parent_chunk_id"].isin(parent_ids)]
    results.append({
        "check": "All children link to a valid parent",
        "passed": len(dangling) == 0,
        "count": len(dangling),
        "pct": round(100 * len(dangling) / max(1, len(child_df)), 2) if len(child_df) else 0.0,
        "severity": "fail" if len(dangling) > 0 else "pass",
    })

    # 6. Empty / whitespace-only chunks
    empty = df[df["text"].str.strip().str.len() < 10]
    results.append({
        "check": "No empty chunks",
        "passed": len(empty) == 0,
        "count": len(empty),
        "pct": round(100 * len(empty) / max(1, len(df)), 2),
        "severity": "fail" if len(empty) > 0 else "pass",
    })

    # 7. Duplicate chunks (by hash)
    dup = df[df.duplicated(subset=["chunk_hash"], keep=False)]
    results.append({
        "check": "No duplicate chunks (by hash)",
        "passed": len(dup) / max(1, len(df)) < 0.01,
        "count": len(dup),
        "pct": round(100 * len(dup) / max(1, len(df)), 2),
        "severity": "warn",
    })

    return pd.DataFrame(results)


checks = run_sanity_checks(df)
print(checks.to_string(index=False))


                              check  passed  count  pct severity
          No chunks under 50 tokens    True      0 0.00     warn
          No chunks over 800 tokens   False     22 6.81     fail
               No orphaned headings    True      0 0.00     warn
                No fractured tables    True      0 0.00     pass
All children link to a valid parent    True      0 0.00     pass
                    No empty chunks    True      0 0.00     pass
      No duplicate chunks (by hash)    True      0 0.00     warn


In [16]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# (a) Token distribution — parents vs children
ax = axes[0, 0]
ax.hist(df[df["is_parent"]]["token_count"], bins=40, alpha=0.6, label="Parents/atomic", color="#2E86AB")
ax.hist(df[~df["is_parent"]]["token_count"], bins=40, alpha=0.6, label="Children", color="#A23B72")
ax.axvline(CHILD_CHUNK_TOKENS, linestyle="--", color="#A23B72", alpha=0.8, label=f"Child target ({CHILD_CHUNK_TOKENS})")
ax.axvline(PARENT_CHUNK_TOKENS, linestyle="--", color="#2E86AB", alpha=0.8, label=f"Parent target ({PARENT_CHUNK_TOKENS})")
ax.axvline(MAX_CHUNK_TOKENS, linestyle=":", color="red", label=f"Hard ceiling ({MAX_CHUNK_TOKENS})")
ax.set_xlabel("Tokens")
ax.set_ylabel("Count")
ax.set_title("Token distribution")
ax.legend(fontsize=8)

# (b) Chunks per page
ax = axes[0, 1]
per_page = df.groupby("page_start").size()
ax.bar(per_page.index, per_page.values, color="#6B8E23")
ax.set_xlabel("Page")
ax.set_ylabel("# chunks")
ax.set_title("Chunks per page (sanity: no page should be empty)")

# (c) Chunk type breakdown
ax = axes[1, 0]
counts = df["chunk_type"].value_counts()
ax.barh(counts.index, counts.values, color="#F18F01")
ax.set_xlabel("Count")
ax.set_title("Chunk type breakdown")

# (d) Top sections by chunk count
ax = axes[1, 1]
top_sections = df["section_path_str"].value_counts().head(10)
ax.barh(range(len(top_sections)), top_sections.values, color="#C73E1D")
ax.set_yticks(range(len(top_sections)))
ax.set_yticklabels([s[:60] + "..." if len(s) > 60 else s for s in top_sections.index], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("# chunks")
ax.set_title("Top 10 sections by chunk count")

plt.tight_layout()
plt.savefig("chunk_diagnostics.png", dpi=110, bbox_inches="tight")
plt.show()
print("Saved: chunk_diagnostics.png")


Saved: /mnt/user-data/outputs/chunk_diagnostics.png


In [17]:
def show_chunk(df: pd.DataFrame, idx: int, max_chars: int = 1500) -> None:
    """Pretty-print a single chunk with all metadata."""
    row = df.iloc[idx]
    print("=" * 80)
    print(f"CHUNK #{idx}  |  id={row['chunk_id'][:8]}…  |  type={row['chunk_type']}  |  tokens={row['token_count']}")
    print("-" * 80)
    print(f"Section:     {row['section_path_str']}")
    print(f"Pages:       {row['page_start']}–{row['page_end']}")
    pid = row['parent_chunk_id']
    pid_display = f"{pid[:8]}…" if (isinstance(pid, str) and pid) else '(this is a parent/atomic)'
    print(f"Parent:      {pid_display}")
    print(f"# children:  {len(row['child_chunk_ids'])}")
    print(f"References:  {row['regulatory_references']}")
    print(f"Hash:        {row['chunk_hash']}")
    print("-" * 80)
    txt = row["text"]
    print(txt if len(txt) <= max_chars else txt[:max_chars] + f"\n… [truncated, {len(txt)-max_chars} more chars]")
    print("=" * 80)


def show_section(df: pd.DataFrame, section_query: str, limit: int = 5) -> None:
    """Show all chunks whose section path contains the query string."""
    mask = df["section_path_str"].str.contains(section_query, case=False, na=False)
    sub = df[mask].head(limit)
    print(f"\nFound {mask.sum()} chunks matching '{section_query}' (showing first {len(sub)})\n")
    for i, (_, row) in enumerate(sub.iterrows()):
        idx = df.index[df["chunk_id"] == row["chunk_id"]][0]
        show_chunk(df, idx, max_chars=600)


# Example uses — adjust indices/queries
show_chunk(df, 0)


CHUNK #0  |  id=e3977b73…  |  type=prose  |  tokens=349
--------------------------------------------------------------------------------
Section:     FATF GUIDANCE > National Risk Assessment
Pages:       1–3
Parent:      (this is a parent/atomic)
# children:  1
References:  ['FATF']
Hash:        6b1e067d41852a67
--------------------------------------------------------------------------------
Guidance

November 2024

The Financial Action Task Force (FATF) is an independent inter-governmental body that develops and promotes
policies to protect the global financial system against money laundering, terrorist financing and the financing of
proliferation of weapons of mass destruction. The FATF Recommendations are recognised as the global anti-money
laundering (AML) and counter-terrorist financing (CFT) standard.

For more information about the FATF, please visit www.fatf-gafi.org

This document and/or any map included herein are without prejudice to the status of or sovereignty over any
ter

In [18]:
# Inspect: any oversized chunks?
oversized = df[df["token_count"] > MAX_CHUNK_TOKENS]
print(f"Oversized chunks: {len(oversized)}")
if len(oversized):
    for idx in oversized.index[:3]:
        show_chunk(df, idx, max_chars=400)

# Inspect: all tables — verify they are intact
print("\n\n" + "=" * 80)
print("ALL TABLES")
print("=" * 80)
tables = df[df["chunk_type"] == "table"]
print(f"Total table chunks: {len(tables)}")
for idx in tables.index[:3]:
    show_chunk(df, idx, max_chars=800)


Oversized chunks: 22
CHUNK #2  |  id=bf713eec…  |  type=prose  |  tokens=1372
--------------------------------------------------------------------------------
Section:     FATF GUIDANCE > Table of Contents
Pages:       3–4
Parent:      (this is a parent/atomic)
# children:  4
References:  ['Annex A', 'Annex B', 'FATF', 'Section 1', 'Section 2', 'Section 3']
Hash:        0f4cf7c825b8dde4
--------------------------------------------------------------------------------
Acronyms ............................................................................................................................................................................... 3

Executive Summary ........................................................................................................................................................... 5

Introduction ......................
… [truncated, 5091 more chars]
CHUNK #16  |  id=203a0fd9…  |  type=prose  |  tokens=1480
---------------------------------------

In [19]:
def search_chunks(df: pd.DataFrame, query: str, limit: int = 5) -> pd.DataFrame:
    """Simple substring search — useful for verifying specific content made it through."""
    mask = df["text"].str.contains(query, case=False, na=False, regex=False)
    hits = df[mask].head(limit)
    print(f"'{query}' found in {mask.sum()} chunks (showing first {len(hits)})")
    return hits[["chunk_id", "chunk_type", "section_path_str", "page_start", "token_count"]]

# Verify specific key concepts from the FATF doc made it through
for q in ["threat", "Recommendation 1", "INR.1", "horizon scanning", "Box 7"]:
    print()
    display(search_chunks(df, q, limit=3))



'threat' found in 134 chunks (showing first 3)


,chunk_id,chunk_type,section_path_str,page_start,token_count
2,bf713eec-d0ee-4ddd-a65d-be7f1349759b,prose,FATF GUIDANCE > Table of Contents,3,1372
4,da08c61b-0198-4820-9ff6-5833c3439bb9,prose,FATF GUIDANCE > Table of Contents,3,411
7,22fb3874-e75c-445d-bcb1-d62c6a407531,prose,FATF GUIDANCE > World Bank Group,6,577



'Recommendation 1' found in 10 chunks (showing first 3)


,chunk_id,chunk_type,section_path_str,page_start,token_count
7,22fb3874-e75c-445d-bcb1-d62c6a407531,prose,FATF GUIDANCE > World Bank Group,6,577
11,82629747-712b-46ca-a2a7-035eac5a1180,prose,FATF GUIDANCE > World Bank Group,6,407
16,203a0fd9-0d5d-437c-a44a-b16c6324e2fc,prose,Section 3: Post-NRA Actions – This section ide...,7,1480



'INR.1' found in 10 chunks (showing first 3)


,chunk_id,chunk_type,section_path_str,page_start,token_count
17,0b766846-283b-44b2-a2c8-76baaf379e39,prose,Section 3: Post-NRA Actions – This section ide...,7,1016
25,93a44964-9494-4fb0-82ba-3ae680336004,prose,Section 3: Post-NRA Actions – This section ide...,7,397
28,71f17487-c86c-45a4-8496-ae82476d68e5,prose,Section 2: Assessing Money Laundering Risks – ...,9,134



'horizon scanning' found in 7 chunks (showing first 3)


,chunk_id,chunk_type,section_path_str,page_start,token_count
2,bf713eec-d0ee-4ddd-a65d-be7f1349759b,prose,FATF GUIDANCE > Table of Contents,3,1372
5,1c6a07d0-12e2-4e24-a0b7-63f5322ee56e,prose,FATF GUIDANCE > Table of Contents,3,411
200,213db803-6bea-4bb8-8556-5e0dedd8f6f0,prose,Section 2: Assessing and Understanding ML Risk...,46,1082



'Box 7' found in 3 chunks (showing first 3)


,chunk_id,chunk_type,section_path_str,page_start,token_count
96,cd314e95-7c2c-4f97-967b-764bba847a63,prose,Section 1: NRA Preparation and Set-up > Even i...,22,985
99,d2dc256d-2a5b-403c-8b97-9a2a5cd8d3d7,prose,Section 1: NRA Preparation and Set-up > Even i...,22,408
102,3b35c783-d939-47de-ba97-55b23351fd60,table,Section 1: NRA Preparation and Set-up > Even i...,24,799


In [20]:
def export_html_report(df: pd.DataFrame, checks: pd.DataFrame, out_path: str) -> None:
    """Write a single-file HTML report for reviewing all chunks in-context."""
    html = ["""<!doctype html><html><head><meta charset="utf-8">
    <title>Chunk Report</title>
    <style>
    body { font-family: -apple-system, sans-serif; max-width: 1100px; margin: 2em auto; padding: 0 1em; color: #222; }
    h1, h2 { color: #2E86AB; }
    .chunk { border: 1px solid #ddd; border-left: 4px solid #2E86AB; padding: 1em; margin: 1em 0; border-radius: 4px; background: #fafafa; }
    .chunk.child { border-left-color: #A23B72; }
    .chunk.table { border-left-color: #F18F01; background: #fff8e7; }
    .chunk.box   { border-left-color: #6B8E23; background: #f4f8e8; }
    .meta { font-size: 0.85em; color: #666; margin-bottom: 0.5em; }
    .meta span { margin-right: 1em; }
    pre { white-space: pre-wrap; word-wrap: break-word; font-family: Menlo, monospace; font-size: 0.9em; background: #fff; padding: 0.5em; border: 1px solid #eee; }
    .check-pass { color: green; font-weight: bold; }
    .check-fail { color: red; font-weight: bold; }
    .check-warn { color: #e67e22; font-weight: bold; }
    table { border-collapse: collapse; }
    th, td { border: 1px solid #ddd; padding: 6px 10px; }
    th { background: #f0f0f0; }
    </style></head><body>"""]

    html.append(f"<h1>Chunk Quality Report</h1>")
    html.append(f"<p><b>Document:</b> {df['doc_title'].iloc[0]}<br>")
    html.append(f"<b>Version:</b> {df['doc_version'].iloc[0]}<br>")
    html.append(f"<b>Total chunks:</b> {len(df)}</p>")

    # Sanity checks summary
    html.append("<h2>Sanity Checks</h2><table><tr><th>Check</th><th>Status</th><th>Count</th><th>%</th></tr>")
    for _, c in checks.iterrows():
        status = "check-pass" if c["passed"] else ("check-fail" if c["severity"] == "fail" else "check-warn")
        mark = "✓" if c["passed"] else "✗"
        html.append(f"<tr><td>{c['check']}</td><td class='{status}'>{mark}</td><td>{c['count']}</td><td>{c['pct']}%</td></tr>")
    html.append("</table>")

    # Chunks (first 50 for a manageable HTML)
    html.append("<h2>Chunks (first 50)</h2>")
    for _, row in df.head(50).iterrows():
        cls = row["chunk_type"]
        if not row["is_parent"] and cls == "prose":
            cls = "child"
        html.append(f"<div class='chunk {cls}'>")
        html.append(f"<div class='meta'>")
        html.append(f"<span><b>ID:</b> {row['chunk_id'][:8]}</span>")
        html.append(f"<span><b>Type:</b> {row['chunk_type']}</span>")
        html.append(f"<span><b>Tokens:</b> {row['token_count']}</span>")
        html.append(f"<span><b>Pages:</b> {row['page_start']}–{row['page_end']}</span>")
        html.append(f"</div>")
        html.append(f"<div class='meta'><b>Section:</b> {row['section_path_str']}</div>")
        if row["regulatory_references"]:
            html.append(f"<div class='meta'><b>Refs:</b> {', '.join(row['regulatory_references'])}</div>")
        # Escape HTML
        safe = (row["text"].replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;"))
        html.append(f"<pre>{safe}</pre>")
        html.append("</div>")

    html.append("</body></html>")
    Path(out_path).write_text("\n".join(html), encoding="utf-8")
    print(f"✅ HTML report written to {out_path}")

export_html_report(df, checks, "chunk_report.html")


✅ HTML report written to /mnt/user-data/outputs/chunk_report.html


## Next Steps

1. **Review the HTML report** (`chunk_report.html`) — spot-check 30-50 random chunks for quality
2. **Run metadata extraction** (next notebook) — pass each chunk through a local LLM (Qwen2.5-7B via vLLM) to extract topic tags, entities, definitions, obligations
3. **Embed & index** — push to Pinecone via OpenAI `text-embedding-3-large` (dims=1024)
4. **Embedding sanity** — project a sample via UMAP in Arize Phoenix to verify semantic separation

**Parallelism tuning tips:**
- Set `CHUNKER_WORKERS` env var to override default (N_cpu - 1)
- For a 10,000-page corpus: run document-level parallelism across multiple PDFs with `ProcessPoolExecutor`, and page-level parallelism within each PDF
- pdfplumber holds the GIL during table extraction — process pool is essential
- tiktoken releases the GIL on the encode path — thread pool is fine for chunk assembly

**Production hardening to add:**
- Swap PyMuPDF → Docling for better table/heading fidelity
- Wire in Langfuse tracing around `process_document()`
- Add Great Expectations suite for the sanity checks (CI gating)
- Implement incremental reprocessing (hash-based: only reprocess changed pages)
